# SPM 405: Machine Learning — Lecture 3 notebook
## Classification

**Dr Charles Ndung'u** · Department of Physics, University of Nairobi

In this notebook you will:
1. Fit **logistic regression** to motor-fault data, first with your own gradient descent and then with scikit-learn.
2. Classify machines from two sensors, and judge the result with the **confusion matrix**, the **threshold** and the **ROC curve**.
3. Compare **logistic regression, LDA, QDA, naive Bayes and KNN** on the same test data.

Reading: ISLP Chapter 4 (4.1–4.5). Run each cell with **Shift + Enter**, from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

rng = np.random.default_rng(405)   # fixed seed: everyone gets the same numbers as the slides

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Part 1: Vibration and motor faults (one input)

200 motors. $X$ = vibration $v$ (RMS velocity, mm/s); $Y = 1$ if the motor is faulty.
The true probability of a fault is $\sigma(-6 + 1.5\,v)$: 50% at 4 mm/s.

In [ ]:
n = 200
vib = rng.uniform(0, 10, n)
fault = (rng.uniform(size=n) < sigmoid(-6 + 1.5 * vib)).astype(int)
print("faulty motors:", fault.sum(), "of", n)

lin = LinearRegression().fit(vib[:, None], fault)
print(f"straight line: y = {lin.intercept_:.2f} + {lin.coef_[0]:.3f} v")
print("its predictions at 0 and 10 mm/s:", lin.predict([[0], [10]]).round(2))

**Question 1.** Why is a straight line a poor model for a probability?

### Logistic regression by gradient descent

Update: $\boldsymbol\beta \leftarrow \boldsymbol\beta + \frac{\alpha}{n}\mathbf X^{\mathsf T}(\mathbf y - \mathbf p)$ — the same rule as linear regression in Lecture 2.
As in Lecture 2, we standardise the input first and convert the coefficients back at the end.

In [ ]:
z = (vib - vib.mean()) / vib.std()
Xz = np.c_[np.ones(n), z]
beta_z = np.zeros(2)
cost = []
for step in range(3001):
    p = sigmoid(Xz @ beta_z)
    cost.append(-np.mean(fault * np.log(p) + (1 - fault) * np.log(1 - p)))   # cross-entropy
    beta_z = beta_z + 1.0 / n * Xz.T @ (fault - p)

b1 = beta_z[1] / vib.std()
b0 = beta_z[0] - b1 * vib.mean()
print(f"gradient descent: p = sigmoid({b0:.2f} + {b1:.3f} v)")
print("cost at the start:", round(cost[0], 3), " at the end:", round(cost[-1], 3))

sk = LogisticRegression(C=np.inf).fit(vib[:, None], fault)    # C=np.inf: no regularisation
print(f"scikit-learn:     p = sigmoid({sk.intercept_[0]:.2f} + {sk.coef_[0, 0]:.3f} v)")

plt.plot(cost); plt.xscale("log"); plt.xlabel("step"); plt.ylabel("cross-entropy"); plt.show()

In [ ]:
X = np.c_[np.ones(n), vib]
p = sigmoid(X @ np.array([b0, b1]))
W = p * (1 - p)
se = np.sqrt(np.diag(np.linalg.inv(X.T @ (W[:, None] * X))))    # standard errors (ISLP Table 4.1)
print("SE:", se.round(3), "  z = estimate / SE:", (np.array([b0, b1]) / se).round(2))
print("50% point:", round(-b0 / b1, 2), "mm/s")
print("odds multiplied by", round(np.exp(b1), 1), "per mm/s")
for v in [2, 4, 6, 8]:
    print(f"P(fault | v = {v}) = {sigmoid(b0 + b1 * v):.3f}")

grid = np.linspace(0, 10, 200)
plt.plot(vib, fault, "|", alpha=0.5)
plt.plot(grid, sigmoid(b0 + b1 * grid), label="logistic regression")
plt.plot(grid, sigmoid(-6 + 1.5 * grid), "k--", label="true probability")
plt.xlabel("vibration v (mm/s)"); plt.ylabel("probability of a fault"); plt.legend(); plt.show()

**Questions**
2. At what vibration level is a motor equally likely to be healthy or faulty? How close is it to the true 4 mm/s?
3. Explain in words what "the odds are multiplied by about 11 per mm/s" means for a maintenance engineer.
4. Try `alpha = 0.01` instead of `1.0` in the gradient descent. What happens to the cost after 3000 steps?

## Part 2: Two sensors on each machine

Inputs: vibration $v$ (mm/s) and bearing temperature $T$ (°C). About 30% of machines are faulty.
Faulty machines vibrate more, run hotter, and vary more.

In [ ]:
def machines(n):
    y = (rng.uniform(size=n) < 0.3).astype(int)          # 1 = faulty
    Xh = rng.multivariate_normal([2.5, 50], [[0.8**2, 0], [0, 5**2]], n)
    Xf = rng.multivariate_normal([4.5, 60], [[1.5**2, 0.8 * 1.5 * 8], [0.8 * 1.5 * 8, 8**2]], n)
    X = np.where(y[:, None] == 1, Xf, Xh)
    X[:, 0] = np.abs(X[:, 0])                             # vibration cannot be negative
    return X, y

X_train, y_train = machines(400)
X_test, y_test = machines(2000)
print("faulty: train", y_train.sum(), "of 400;  test", y_test.sum(), "of 2000")

logit = LogisticRegression(C=np.inf).fit(X_train, y_train)
b0 = logit.intercept_[0]; b1, b2 = logit.coef_[0]
print(f"p = sigmoid({b0:.2f} + {b1:.3f} v + {b2:.4f} T)")
print(f"boundary: T = {-b0 / b2:.1f} + ({-b1 / b2:.2f}) v")

plt.scatter(*X_train[y_train == 0].T, s=8, label="healthy")
plt.scatter(*X_train[y_train == 1].T, s=8, label="faulty")
vv = np.linspace(0, 8, 2)
plt.plot(vv, -b0 / b2 - b1 / b2 * vv, "k", label="p = 0.5")
plt.ylim(30, 85); plt.xlabel("vibration v (mm/s)"); plt.ylabel("bearing temperature T (°C)"); plt.legend(); plt.show()

In [ ]:
p_test = logit.predict_proba(X_test)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_test, p_test >= 0.5).ravel()
print("confusion matrix (threshold 0.5):")
print(f"   truly faulty : TP = {tp:4d}   FN = {fn:4d}")
print(f"   truly healthy: FP = {fp:4d}   TN = {tn:4d}")

for th in [0.2, 0.5, 0.8]:
    tn, fp, fn, tp = confusion_matrix(y_test, p_test >= th).ravel()
    print(f"threshold {th}: accuracy {(tp + tn) / len(y_test):.3f}  recall {tp / (tp + fn):.3f}  "
          f"precision {tp / (tp + fp):.3f}  specificity {tn / (tn + fp):.3f}  missed {fn}  false alarms {fp}")

fpr, tpr, _ = roc_curve(y_test, p_test)
plt.plot(fpr, tpr, label=f"logistic (AUC = {roc_auc_score(y_test, p_test):.2f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("false positive rate"); plt.ylabel("recall"); plt.legend(); plt.show()

**Questions**
5. A missed fault costs a factory KSh 500 000 (a breakdown); a false alarm costs KSh 20 000 (an inspection).
   Using the numbers above, which of the three thresholds gives the lowest total cost?
6. What error rate would you get by always saying "healthy"? Why is accuracy alone misleading here?

## Part 3: Comparing five classifiers

In [ ]:
scaler = StandardScaler().fit(X_train)                  # fitted on the training data only
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

models = {
    "Logistic regression": (LogisticRegression(C=np.inf), False),
    "LDA": (LinearDiscriminantAnalysis(), False),
    "QDA": (QuadraticDiscriminantAnalysis(), False),
    "Naive Bayes": (GaussianNB(), False),
    "KNN K=1": (KNeighborsClassifier(1), True),
    "KNN K=15": (KNeighborsClassifier(15), True),
    "KNN K=15, not standardised": (KNeighborsClassifier(15), False),
}
print(f"{'always say healthy':28s} test error {y_test.mean():.3f}")
for name, (model, standardise) in models.items():
    A, B = (Z_train, Z_test) if standardise else (X_train, X_test)
    model.fit(A, y_train)
    prob = model.predict_proba(B)[:, 1]
    print(f"{name:28s} test error {np.mean((prob >= 0.5) != y_test):.3f}   AUC {roc_auc_score(y_test, prob):.3f}   "
          f"train error {np.mean(model.predict(A) != y_train):.3f}")

In [ ]:
lda = models["LDA"][0]
print("LDA priors (healthy, faulty):", lda.priors_.round(3))
print("LDA class means (v, T):\n", lda.means_.round(2))

qda = models["QDA"][0]
vg, Tg = np.meshgrid(np.linspace(0, 9, 200), np.linspace(30, 85, 200))
P = qda.predict_proba(np.c_[vg.ravel(), Tg.ravel()])[:, 1].reshape(vg.shape)
plt.scatter(*X_train[y_train == 0].T, s=8, label="healthy")
plt.scatter(*X_train[y_train == 1].T, s=8, label="faulty")
plt.contour(vg, Tg, P, levels=[0.5], colors="red")
plt.plot(vv, -b0 / b2 - b1 / b2 * vv, "k")
plt.xlabel("vibration v (mm/s)"); plt.ylabel("bearing temperature T (°C)")
plt.title("black: logistic boundary, red: QDA boundary"); plt.legend(); plt.show()

**Questions**
7. Which method has the lowest test error? Why might QDA suit these machines? (Look at the spread of the two classes.)
8. KNN with $K = 1$ has training error 0. Is it the best model? Explain with the bias–variance trade-off from Lecture 1.
9. Why does standardising the inputs matter for KNN but not for LDA or logistic regression?
10. **Your group project:** if your output is a category, which classifier would you try first, and which measure (recall, precision, AUC) matters most for your problem?

## Your turn: group tasks

Work on these in your group. Write the code in the empty cells and answer each question in a text cell below it,
in your own words. Submit **one notebook per group** on Google Classroom, with every cell run.
Every member must be able to explain every line.

The tasks reuse `machines`, `logit`, `X_train`, `y_train`, `X_test`, `y_test` and `scaler` from above, so run the whole notebook first.

### Task 1: Choosing $K$ for KNN without touching the test data

The test data must only be used **once**, at the very end. To choose $K$ we hold back part of the training data instead:
a **validation set**.

(a) Split the 400 standardised training machines (`Z_train`, `y_train`) into 300 for fitting and 100 for validation.

(b) For $K = 1, 2, \dots, 60$, fit KNN on the 300 and compute the error rate on the 100. Plot the validation error against $K$.

(c) Choose $K$. Refit KNN with that $K$ on all 400 training machines and report its **test** error. Compare it with the table in Part 3.
Which values of $K$ overfit, and which underfit?

In [ ]:
# Task 1: your code here

### Task 2: The cheapest threshold

This time a missed fault costs KSh 200 000 (a repair after a breakdown) and a false alarm costs KSh 20 000 (an inspection).

(a) Make 5000 new machines with `machines(5000)` as a **validation** set. Use the logistic model `logit` to get their fault probabilities.

(b) For thresholds from 0.01 to 0.99 in steps of 0.01, compute the total cost on the validation machines. Plot cost against threshold
and find the cheapest threshold.

(c) Apply that threshold to the 2000 test machines. Give the confusion matrix and the total cost, and compare with the cost at threshold 0.5.
Explain to a factory manager, in two or three sentences, why the best threshold is far from 0.5.

(d) Theory says that, if the probabilities are right, the cheapest threshold is
$\dfrac{\text{cost of a false alarm}}{\text{cost of a false alarm} + \text{cost of a miss}}$. Compute it and compare with your answer to (b).

In [ ]:
# Task 2: your code here

### Task 3: Three classes — which fault is it?

A maintenance team wants to know not only *whether* a motor is faulty but *what* is wrong. Each motor is **healthy** (0),
**unbalanced** (1) or has **bearing wear** (2). Two features come from its vibration spectrum: the amplitude at the running speed
$a_1$ (mm/s) and the high-frequency energy $h$ (arbitrary units). The cell below makes the data; do not change it.

(a) Plot the training motors, coloured by class.

(b) Fit logistic regression (scikit-learn handles three classes by itself) and LDA on the training motors. Give the test error rate
and the $3 \times 3$ confusion matrix of each.

(c) Which two classes are most often confused? Looking at your plot, why?

In [ ]:
rng3 = np.random.default_rng(4054)                # task data: do not change
def motors(n):
    y = rng3.integers(0, 3, n)                       # 0 healthy, 1 unbalanced, 2 bearing wear
    means = np.array([[1.0, 1.0], [2.3, 1.2], [1.6, 3.0]])
    X = means[y] + rng3.normal(0, [0.6, 0.6], (n, 2))
    return X, y
M_train, c_train = motors(300)
M_test, c_test = motors(3000)

In [ ]:
# Task 3: your code here